# MP2 Part 1: Commit retrieval (rkabarwa)
Retrieves all commits for my 10 assigned projects from World of Code and basic repository info from GitHub.

In [1]:
# !python -m pip install -U python-woc pandas matplotlib tqdm requests tabulate
import json, os, time
import pandas as pd
import requests
from tqdm import tqdm
from woc.remote import WocMapsRemote

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
netid = 'rkabarwa'

projects = pd.read_csv('net2prj.csv')
projects = projects[projects.netID == netid].reset_index(drop=True)
projects

,netID,WoC,GH
0,rkabarwa,dlr-rm_blenderproc,https://github.com/DLR-RM/BlenderProc
1,rkabarwa,riatelab_cartography,https://github.com/riatelab/cartography
2,rkabarwa,patrickzib_sfa,https://github.com/patrickzib/SFA
3,rkabarwa,openexoplanetcatalogue_open_exoplanet_catalogue,https://github.com/OpenExoplanetCatalogue/open...
4,rkabarwa,snakemake-workflows_rna-seq-kallisto-sleuth,https://github.com/snakemake-workflows/rna-seq...
5,rkabarwa,brianmanderson_dicom_utilities,https://github.com/brianmanderson/dicom_utilities
6,rkabarwa,iraf-community_iraf,https://github.com/iraf-community/iraf
7,rkabarwa,dgerosa_precession,https://github.com/dgerosa/precession
8,rkabarwa,plasmapy_plasmapy,https://github.com/PlasmaPy/PlasmaPy
9,rkabarwa,iqusoft_intel-qs,https://github.com/iqusoft/intel-qs


## Step 1: commit sha1s for each project (p2c)

In [2]:
dfs = []
for prj in projects.WoC:
    shas = woc.get_values('p2c', prj)
    print(prj, len(shas))
    dfs.append(pd.DataFrame({'sha1': shas, 'project': prj}))
    time.sleep(1)
df = pd.concat(dfs, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
print('total', len(df))

dlr-rm_blenderproc 5512


riatelab_cartography 737


patrickzib_sfa 468


openexoplanetcatalogue_open_exoplanet_catalogue 7241


snakemake-workflows_rna-seq-kallisto-sleuth 1421


...

iraf-community_iraf 2783


dgerosa_precession 1191


plasmapy_plasmapy 25704


iqusoft_intel-qs 543


total 46007


## Step 2: commit content in batches of 40 (under the 50 limit), 1s wait between requests
Results are cached in `commit_cache.jsonl` so the cell can be re-run after an interruption without re-downloading.

In [3]:
CACHE = 'commit_cache.jsonl'
done = {}
if os.path.exists(CACHE):
    with open(CACHE) as f:
        for line in f:
            r = json.loads(line)
            done[r['commit']] = r

todo = [s for s in df.sha1.unique() if s not in done]
chunks = [todo[i:i+40] for i in range(0, len(todo), 40)]
missing = {}
with open(CACHE, 'a') as f:
    for chunk in tqdm(chunks):
        for attempt in range(10):
            try:
                res, err = woc.get_values_many('commit.tch', chunk)
                break
            except Exception as e:
                print('retrying after error:', e)
                time.sleep(65)  # WoC asks clients to wait 60s when rate limited
        else:
            raise RuntimeError('batch failed 10 times')
        missing.update(err or {})
        for sha, v in res.items():
            c = v[0]  # [tree, parents, [author, time, tz], [committer, time, tz], message]
            r = {'commit': sha, 'author': c[2][0], 'time': int(c[2][1]), 'message': c[4]}
            done[sha] = r
            f.write(json.dumps(r) + '\n')
        time.sleep(1)

print('retrieved', len(done), 'of', df.sha1.nunique(), 'unique commits;',
      len(missing), 'not found in WoC commit.tch in this run')

  0%|          | 0/1135 [00:00<?, ?it/s]

  0%|          | 1/1135 [00:01<27:12,  1.44s/it]

  0%|          | 2/1135 [00:02<25:16,  1.34s/it]

  0%|          | 3/1135 [00:03<23:56,  1.27s/it]

  0%|          | 4/1135 [00:05<23:29,  1.25s/it]

...

100%|█████████▉| 1133/1135 [28:23<00:02,  1.19s/it]

100%|█████████▉| 1134/1135 [28:24<00:01,  1.19s/it]

100%|██████████| 1135/1135 [28:25<00:00,  1.16s/it]

100%|██████████| 1135/1135 [28:25<00:00,  1.50s/it]

retrieved 45985 of 46007 unique commits; 22 not found in WoC commit.tch in this run


## Step 3: write `rkabarwa_project_summary.csv` (semicolon separated)

In [4]:
dfc = df.merge(pd.DataFrame(done.values()), left_on='sha1', right_on='commit')
summary = dfc[['project', 'sha1', 'author', 'time', 'message']]
summary.columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
summary.to_csv(netid + '_project_summary.csv', sep=';', index=False)
print(len(summary), 'rows written')
summary.head()

45985 rows written


,project_wocid,commit_sha1,author,time,commit message
0,dlr-rm_blenderproc,0003f0a70b91315ba228f3adf329bf06cf91641a,Klaus Strobl <44800028+5trobl@users.noreply.gi...,1676903306,chore(readme): Update links
1,dlr-rm_blenderproc,001167c20e8727ad888d68483dd087196a9fc6d1,Maximilian Denninger <maximilian.denninger@dlr...,1587566621,feat(PartSphere): add a vector to specify the ...
2,dlr-rm_blenderproc,0012deb4b57bf2d5a967693bf04757ee0915cb86,Dmitry Olefir <dmitry.olefir@dlr.de>,1579791599,example refactoring\n
3,dlr-rm_blenderproc,001d788bf307f82225a4d20643c1ba216fa2c028,Maximilian Denninger <maximilian.denninger@dlr...,1587550025,doc(ShapeNet/README): add proper README for ba...
4,dlr-rm_blenderproc,0022f326761cbe375582bebc2d36934ec182cde4,Martin Sundermeyer <martin.sundermeyer@dlr.de>,1590422965,started elevation uniform\n


## Step 4: WoC stats per project

In [5]:
stats = summary.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'count'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max'))
stats['min_time'] = pd.to_datetime(stats.min_time, unit='s')
stats['max_time'] = pd.to_datetime(stats.max_time, unit='s')
stats = stats.reindex(projects.WoC)
print(stats.to_markdown())

| WoC                                             |   ncommits |   nauthors | min_time            | max_time            |
|:------------------------------------------------|-----------:|-----------:|:--------------------|:--------------------|
| dlr-rm_blenderproc                              |       5512 |        152 | 2019-06-28 09:32:06 | 2026-03-15 01:03:33 |
| riatelab_cartography                            |        737 |         20 | 2015-06-11 08:27:11 | 2025-07-23 07:06:59 |
| patrickzib_sfa                                  |        466 |         13 | 2016-02-22 14:38:46 | 2025-05-02 13:28:17 |
| openexoplanetcatalogue_open_exoplanet_catalogue |       7241 |         90 | 2011-01-11 03:03:32 | 2025-11-05 07:01:15 |
| snakemake-workflows_rna-seq-kallisto-sleuth     |       1420 |         47 | 2018-12-12 10:51:17 | 2025-10-31 11:53:49 |
| brianmanderson_dicom_utilities                  |        407 |         14 | 2019-10-09 18:50:49 | 2021-07-22 19:01:38 |
| iraf-community_iraf   

## Step 5: GitHub stats (stars, forks, last commit date)

In [6]:
gh = []
for wid, url in zip(projects.WoC, projects.GH):
    repo = url.replace('https://github.com/', '')
    r = requests.get(f'https://api.github.com/repos/{repo}').json()
    c = requests.get(f'https://api.github.com/repos/{repo}/commits?per_page=1').json()
    gh.append({'project': wid, 'repo': repo,
               'nstars': r['stargazers_count'], 'nforks': r['forks_count'],
               'lastGHCommitDate': c[0]['commit']['committer']['date'][:10]})
    time.sleep(1)
gh = pd.DataFrame(gh)
print(gh.to_markdown(index=False))

| project                                         | repo                                            |   nstars |   nforks | lastGHCommitDate   |
|:------------------------------------------------|:------------------------------------------------|---------:|---------:|:-------------------|
| dlr-rm_blenderproc                              | DLR-RM/BlenderProc                              |     3730 |      519 | 2026-01-07         |
| riatelab_cartography                            | riatelab/cartography                            |      403 |       44 | 2025-07-23         |
| patrickzib_sfa                                  | patrickzib/SFA                                  |      321 |       67 | 2025-05-02         |
| openexoplanetcatalogue_open_exoplanet_catalogue | OpenExoplanetCatalogue/open_exoplanet_catalogue |     1128 |      224 | 2026-09-28         |
| snakemake-workflows_rna-seq-kallisto-sleuth     | snakemake-workflows/rna-seq-kallisto-sleuth     |       69 |       48 | 2026-0

## Part 1 results

WoC columns come from the retrieved commits (author time, UTC); GitHub columns were collected on 2026-09-28.

| Project | WoC commits | WoC authors | Min time | Max time | GitHub stars | GitHub forks | Last GitHub commit |
|---|---:|---:|---|---|---:|---:|---|
| dlr-rm_blenderproc | 5,512 | 152 | 2019-06-28 | 2026-03-15 | 3,730 | 519 | 2026-01-07 |
| riatelab_cartography | 737 | 20 | 2015-06-11 | 2025-07-23 | 403 | 44 | 2025-07-23 |
| patrickzib_sfa | 466 | 13 | 2016-02-22 | 2025-05-02 | 321 | 67 | 2025-05-02 |
| openexoplanetcatalogue_open_exoplanet_catalogue | 7,241 | 90 | 2011-01-11 | 2025-11-05 | 1,128 | 224 | 2026-09-28 |
| snakemake-workflows_rna-seq-kallisto-sleuth | 1,420 | 47 | 2018-12-12 | 2025-10-31 | 69 | 48 | 2026-08-17 |
| brianmanderson_dicom_utilities | 407 | 14 | 2019-10-09 | 2021-07-22 | 108 | 32 | 2026-09-08 |
| iraf-community_iraf | 2,782 | 36 | 1989-02-22 | 2025-11-02 | 180 | 54 | 2026-09-25 |
| dgerosa_precession | 1,191 | 18 | 2015-11-12 | 2025-08-26 | 42 | 18 | 2026-06-06 |
| plasmapy_plasmapy | 25,686 | 355 | 2015-11-24 | 2026-04-17 | 714 | 374 | 2026-09-28 |
| iqusoft_intel-qs | 543 | 58 | 2017-11-06 | 2025-09-22 | 264 | 76 | 2026-09-11 |